# Multi-Regional AeroMAPS - All Regions (Simple)

This notebook demonstrates running multiple regions sequentially using `MultiRegionalProcess`.

## Configuration

- **Mode**: `separate_processes` (sequential execution)
- **Regions**: 20 world regions (excluding France and Germany detailed configs)
- **Settings**: Default AeroMAPS parameters for all regions

## 0. Prepare Region Data

Generate partitioning files and config.yaml for each region from their CSV data.

In [ ]:
from pathlib import Path
from aeromaps.utils.functions import create_partitioning
import shutil

# Reference files from region_default
default_folder = Path("data/region_default")
energy_carriers_file = default_folder / "energy_carriers_data.yaml"

# All region folders (with actual folder names including typos and asterisks)
region_folders = [
    "region_af_dom", "region_af_int", "region_as_dom", "region_bras_dom",
    "region_eu_dom*", "region_eu_int*", "region_oc_dom", "region_oc_int",
    "region_other_as_int", "region_other_eur_dom", "region_other_eur_int",
    "region_other_nam_dom", "region_other_nam_int", "region_other_sam_dom",
    "region_sin_int", "region_uk_dom*", "region_uk_int*",
    "region_usa_dom", "region_usa_int", "region_sam_int"
]

data_path = Path("data")

for folder_name in region_folders:
    folder_path = data_path / folder_name
    
    if not folder_path.exists():
        print(f"⚠ Folder not found: {folder_name}")
        continue
    
    csv_file = folder_path / "dataframe_aeromaps.csv"
    if not csv_file.exists():
        print(f"⚠ CSV not found: {csv_file}")
        continue
    
    # 1. Generate partitioning from CSV (creates partitioning_updated_inputs.json)
    create_partitioning(file=str(csv_file), path=str(folder_path))
    
    # 2. Copy energy_carriers_data.yaml if not present
    dest_energy = folder_path / "energy_carriers_data.yaml"
    if not dest_energy.exists():
        shutil.copy(energy_carriers_file, dest_energy)
    
    # 3. Create config.yaml if not present (same format as region_france)
    config_file = folder_path / "config.yaml"
    if not config_file.exists():
        config_content = '''data:
  inputs:
    partitioning_data_file: "./partitioning_updated_inputs.json"
  outputs:
    json_outputs_file: "./outputs.json"

models:
  energy:
    energy_carriers_model_data_file: "./energy_carriers_data.yaml"
    resources_model_data_file: default
    processes_model_data_file: default

  standards:
    - models_traffic
    - models_efficiency_bottom_up
    - models_energy_without_fuel_effect
    - models_energy_cost
    - models_offset
    - models_emissions
'''
        config_file.write_text(config_content)
        
    print(f"✓ Prepared: {folder_name}")

print(f"\n✓ All {len(region_folders)} regions prepared")

## 1. Create Multi-Regional Process

In [ ]:
from aeromaps import create_multi_regional_process

# Create the multi-regional process
process = create_multi_regional_process(
    configuration_file="regionalisation_all_regions.yaml", disable_execution_statistics=True
)

print(f"✓ Process created")
print(f"  Mode: {process.execution_mode}")
print(f"  Regions ({len(process.list_regions())}): {process.list_regions()}")

## 2. Execute Sequential Computation

Run all regions sequentially with progress bar.

In [ ]:
import time

# Sequential execution (default)
start_time = time.time()
process.compute(parallel=False)  # Sequential mode
elapsed = time.time() - start_time

print(f"\n✓ Computation complete in {elapsed:.2f} seconds")

## 3. View Outputs

All outputs are stored with namespace prefixes in `data["vector_outputs"]`.

In [ ]:
# Global aggregated outputs
global_outputs = process.get_global_outputs()
print(f"Global outputs shape: {global_outputs.shape}")
global_outputs.head()

In [ ]:
# All outputs with namespaces
print(f"Total columns in vector_outputs: {len(process.data['vector_outputs'].columns)}")
process.data['vector_outputs'].columns.tolist()[:20]  # First 20

In [ ]:
# Regional outputs for a specific region
eu_dom = process.get_regional_outputs("EU_DOM")
print(f"EU_DOM outputs shape: {eu_dom.shape}")
eu_dom.head()

## 4. Emissions Analysis

Compare CO2 emissions metrics across all regions.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: CO2 emissions including energy
ax1 = axes[0]
for region_id in process.list_regions():
    col = f"{region_id}:co2_emissions_including_energy"
    if col in process.data["vector_outputs"].columns:
        print("plotting", region_id)
        print(process.data["vector_outputs"][col])
        process.data["vector_outputs"][col].plot(ax=ax1, label=region_id, alpha=0.7)

# Add global total
global_col = "overall:co2_emissions_including_energy"
if global_col in process.data["vector_outputs"].columns:
    process.data["vector_outputs"][global_col].plot(
        ax=ax1, label="GLOBAL", linewidth=3, linestyle="--", color="black"
    )

ax1.set_xlabel("Year")
ax1.set_ylabel("CO2 Emissions (Mt)")
ax1.set_title("CO2 Emissions Including Energy by Region")
ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
ax1.grid(True, alpha=0.3)

# Plot 2: CO2 emissions 2019 technology
ax2 = axes[1]
for region_id in process.list_regions():
    col = f"{region_id}:co2_emissions_2019technology"
    if col in process.data["vector_outputs"].columns:
        process.data["vector_outputs"][col].plot(ax=ax2, label=region_id, alpha=0.7)

# Add global total
global_col = "overall:co2_emissions_2019technology"
if global_col in process.data["vector_outputs"].columns:
    process.data["vector_outputs"][global_col].plot(
        ax=ax2, label="GLOBAL", linewidth=3, linestyle="--", color="black"
    )

ax2.set_xlabel("Year")
ax2.set_ylabel("CO2 Emissions (Mt)")
ax2.set_title("CO2 Emissions 2019 Technology (BAU) by Region")
ax2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
process.data["vector_outputs"]["USA_INT:energy_consumption"].loc[2025:2050]/1e9